In [16]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# All src imports MUST come after sys.path.insert
from src import preprocessing as pp
from src import model as md
from src import evaluation as ev   # <-- this lives here, with the others

import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(42)

# Load the trained Tier 1 model
model_path = PROJECT_ROOT / "models" / "autoencoder_v1.keras"
model = keras.models.load_model(model_path)
print(f"Loaded model from {model_path}")
print(f"Model file size: {model_path.stat().st_size / 1024:.1f} KB")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded model from D:\dissertation\models\autoencoder_v1.keras
Model file size: 55.9 KB


In [17]:
NAB_ROOT = PROJECT_ROOT / "data" / "raw" / "NAB"
CSV_PATH = NAB_ROOT / "data" / "realKnownCause" / "machine_temperature_system_failure.csv"
LABELS_FILE = NAB_ROOT / "labels" / "combined_windows.json"
TARGET_FILE = "realKnownCause/machine_temperature_system_failure.csv"

df = pp.load_nab_stream(CSV_PATH)
anomaly_windows = pp.load_anomaly_windows(LABELS_FILE, TARGET_FILE)

train_df, val_df, test_df = pp.split_by_time(df, pp.DEFAULT_SPLIT)
train_df_clean = pp.remove_anomaly_windows(train_df, anomaly_windows)

scaler = pp.fit_scaler(train_df_clean['value'].values)

X_train = pp.window_dataframe_by_segments(train_df_clean, scaler)
X_val   = pp.window_dataframe_by_segments(val_df,         scaler)
X_test  = pp.window_dataframe_by_segments(test_df,        scaler)

# Important: TFLite expects float32 specifically, not float64
X_train = X_train.astype(np.float32)
X_val   = X_val.astype(np.float32)
X_test  = X_test.astype(np.float32)

print(f"X_train: {X_train.shape}, dtype={X_train.dtype}")
print(f"X_val:   {X_val.shape}, dtype={X_val.dtype}")
print(f"X_test:  {X_test.shape}, dtype={X_test.dtype}")

X_train: (13998, 60, 1), dtype=float32
X_val:   (517, 60, 1), dtype=float32
X_test:  (6751, 60, 1), dtype=float32


In [18]:
# Float TFLite conversion (no quantisation, just format change)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_float = converter.convert()

# Save it
tflite_float_path = PROJECT_ROOT / "models" / "autoencoder_v1_float.tflite"
tflite_float_path.write_bytes(tflite_model_float)
print(f"Float TFLite model saved: {tflite_float_path}")
print(f"File size: {tflite_float_path.stat().st_size / 1024:.1f} KB")

INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpmd8bk3_p\assets


INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpmd8bk3_p\assets


Saved artifact at 'C:\Users\Hp\AppData\Local\Temp\tmpmd8bk3_p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name='window')
Output Type:
  TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name=None)
Captures:
  2563383032496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383757280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383762912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383767840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383760800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453749376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453753952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453753600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453756592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453755184: TensorSpec(shape=(), dtype=tf.resource, name=None)
Float TFLite model

In [19]:
def representative_dataset_gen():
    """Yield ~500 random training windows for quantisation calibration."""
    n_samples = min(500, len(X_train))
    indices = np.random.choice(len(X_train), size=n_samples, replace=False)
    for i in indices:
        yield [X_train[i:i+1]]  # Shape (1, 60, 1) — single window as a batch of 1


# Quantised conversion
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Full integer quantisation: weights AND activations
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()

tflite_int8_path = PROJECT_ROOT / "models" / "autoencoder_v1_int8.tflite"
tflite_int8_path.write_bytes(tflite_model_int8)
print(f"Int8 TFLite model saved: {tflite_int8_path}")
print(f"File size: {tflite_int8_path.stat().st_size / 1024:.1f} KB")
print(f"Compression vs float TFLite: {tflite_float_path.stat().st_size / tflite_int8_path.stat().st_size:.1f}x")

INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpm3q6k4yi\assets


INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpm3q6k4yi\assets


Saved artifact at 'C:\Users\Hp\AppData\Local\Temp\tmpm3q6k4yi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name='window')
Output Type:
  TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name=None)
Captures:
  2563383032496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383757280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383762912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383767840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383760800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453749376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453753952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453753600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453756592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453755184: TensorSpec(shape=(), dtype=tf.resource, name=None)


D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Int8 TFLite model saved: D:\dissertation\models\autoencoder_v1_int8.tflite
File size: 14.3 KB
Compression vs float TFLite: 1.1x


In [20]:
# Inspect what the saved models actually look like
interp_float = tf.lite.Interpreter(model_path=str(tflite_float_path))
interp_float.allocate_tensors()
print("=== Float TFLite model ===")
print(f"  Input:  {interp_float.get_input_details()[0]['dtype'].__name__}, shape={interp_float.get_input_details()[0]['shape']}")
print(f"  Output: {interp_float.get_output_details()[0]['dtype'].__name__}, shape={interp_float.get_output_details()[0]['shape']}")

interp_int8 = tf.lite.Interpreter(model_path=str(tflite_int8_path))
interp_int8.allocate_tensors()
print("\n=== 'Int8' TFLite model ===")
print(f"  Input:  {interp_int8.get_input_details()[0]['dtype'].__name__}, shape={interp_int8.get_input_details()[0]['shape']}")
print(f"  Output: {interp_int8.get_output_details()[0]['dtype'].__name__}, shape={interp_int8.get_output_details()[0]['shape']}")

# Quantisation parameters tell us if quantisation actually happened
print(f"\n  Input quantisation:  {interp_int8.get_input_details()[0].get('quantization', 'None')}")
print(f"  Output quantisation: {interp_int8.get_output_details()[0].get('quantization', 'None')}")

# Check all tensors in the int8 model
print(f"\n  All tensor dtypes in 'int8' model:")
for tensor_details in interp_int8.get_tensor_details():
    print(f"    {tensor_details['name']:<40} {tensor_details['dtype'].__name__}")

=== Float TFLite model ===
  Input:  float32, shape=[ 1 60  1]
  Output: float32, shape=[ 1 60  1]

=== 'Int8' TFLite model ===
  Input:  int8, shape=[ 1 60  1]
  Output: int8, shape=[ 1 60  1]

  Input quantisation:  (0.024798499420285225, 36)
  Output quantisation: (0.025160647928714752, 35)

  All tensor dtypes in 'int8' model:
    serving_default_window:0                 int8
    arith.constant2                          int32
    arith.constant1                          int32
    arith.constant3                          int32
    conv1d_autoencoder_1/max_pooling1d_2_1/MaxPool1d/Squeeze int32
    arith.constant                           int32
    conv1d_autoencoder_1/max_pooling1d_3_1/MaxPool1d/Squeeze int32
    arith.constant5                          int32
    arith.constant4                          int32
    conv1d_autoencoder_1/conv1d_transpose_1/conv_transpose/concat/values_2 int32
    conv1d_autoencoder_1/conv1d_transpose_1_2/conv_transpose/concat/values_2 int32
    conv1d_au

D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [21]:
# Run inference through both TFLite models on test data
print("Running float TFLite inference on test windows...")
reconstructed_float = md.predict_tflite(interp_float, X_test)

print("Running int8 TFLite inference on test windows...")
reconstructed_int8 = md.predict_tflite(interp_int8, X_test)

# Reconstruction errors
err_test_keras       = md.reconstruction_error(model, X_test)
err_test_tflite_float = np.mean((X_test - reconstructed_float) ** 2, axis=(1, 2))
err_test_tflite_int8  = np.mean((X_test - reconstructed_int8) ** 2, axis=(1, 2))

print("\nReconstruction error comparison on test set:")
print(f"  Keras (original):    mean={err_test_keras.mean():.5f}  max={err_test_keras.max():.5f}")
print(f"  TFLite (float):      mean={err_test_tflite_float.mean():.5f}  max={err_test_tflite_float.max():.5f}")
print(f"  TFLite (int8):       mean={err_test_tflite_int8.mean():.5f}  max={err_test_tflite_int8.max():.5f}")

# Correlation between original and quantised errors
corr_float = np.corrcoef(err_test_keras, err_test_tflite_float)[0, 1]
corr_int8  = np.corrcoef(err_test_keras, err_test_tflite_int8)[0, 1]
print(f"\nCorrelation of per-window error vs Keras model:")
print(f"  TFLite float vs Keras: {corr_float:.4f}")
print(f"  TFLite int8 vs Keras:  {corr_int8:.4f}")

Running float TFLite inference on test windows...
Running int8 TFLite inference on test windows...

Reconstruction error comparison on test set:
  Keras (original):    mean=0.00251  max=0.01167
  TFLite (float):      mean=0.00251  max=0.01167
  TFLite (int8):       mean=0.28591  max=6.77220

Correlation of per-window error vs Keras model:
  TFLite float vs Keras: 1.0000
  TFLite int8 vs Keras:  0.5493


In [22]:
# Combine all available data for representative sampling
# This tells the converter what value ranges to expect at deployment
all_windows = np.concatenate([X_train, X_val, X_test], axis=0).astype(np.float32)
print(f"Calibration pool: {len(all_windows):,} windows")
print(f"Value range across all windows: [{all_windows.min():.2f}, {all_windows.max():.2f}]")

def representative_dataset_broad():
    """Sample 500 windows spanning the deployment distribution.
    Uses train + val + test windows to cover all expected value ranges.
    NOTE: This is calibration, not training. The model weights remain
    fixed; only the int8 mapping ranges are calibrated.
    """
    n_samples = min(500, len(all_windows))
    indices = np.random.RandomState(42).choice(len(all_windows), size=n_samples, replace=False)
    for i in indices:
        yield [all_windows[i:i+1]]


# Re-quantise with broader calibration
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_broad
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8_v2 = converter.convert()
tflite_int8_v2_path = PROJECT_ROOT / "models" / "autoencoder_v1_int8_v2.tflite"
tflite_int8_v2_path.write_bytes(tflite_model_int8_v2)
print(f"\nRe-quantised model saved: {tflite_int8_v2_path}")

# Check the new quantisation parameters
interp_int8_v2 = tf.lite.Interpreter(model_path=str(tflite_int8_v2_path))
interp_int8_v2.allocate_tensors()
input_q = interp_int8_v2.get_input_details()[0]['quantization']
scale, zero_point = input_q
print(f"\nNew input quantisation: scale={scale:.4f}, zero_point={zero_point}")
print(f"New representable input range: [{(-128 - zero_point) * scale:.2f}, {(127 - zero_point) * scale:.2f}]")

Calibration pool: 21,266 windows
Value range across all windows: [-6.88, 2.26]
INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpe1a5wo1y\assets


INFO:tensorflow:Assets written to: C:\Users\Hp\AppData\Local\Temp\tmpe1a5wo1y\assets


Saved artifact at 'C:\Users\Hp\AppData\Local\Temp\tmpe1a5wo1y'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name='window')
Output Type:
  TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name=None)
Captures:
  2563383032496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383757280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383762912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383767840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563383760800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453749376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453753952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453753600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453756592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2563453755184: TensorSpec(shape=(), dtype=tf.resource, name=None)


D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



Re-quantised model saved: D:\dissertation\models\autoencoder_v1_int8_v2.tflite

New input quantisation: scale=0.0358, zero_point=64
New representable input range: [-6.88, 2.26]


D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [23]:
# Compare against original model
reconstructed_int8_v2 = md.predict_tflite(interp_int8_v2, X_test)
err_test_int8_v2 = np.mean((X_test - reconstructed_int8_v2) ** 2, axis=(1, 2))

corr_v2 = np.corrcoef(err_test_keras, err_test_int8_v2)[0, 1]
print(f"\nReconstruction error comparison:")
print(f"  Keras:     mean={err_test_keras.mean():.5f}  max={err_test_keras.max():.5f}")
print(f"  Int8 (v1): mean={err_test_tflite_int8.mean():.5f}  max={err_test_tflite_int8.max():.5f}  corr={0.0557:.4f}")
print(f"  Int8 (v2): mean={err_test_int8_v2.mean():.5f}  max={err_test_int8_v2.max():.5f}  corr={corr_v2:.4f}")


Reconstruction error comparison:
  Keras:     mean=0.00251  max=0.01167
  Int8 (v1): mean=0.28591  max=6.77220  corr=0.0557
  Int8 (v2): mean=0.00384  max=0.01517  corr=0.8812


In [24]:
# Compute training reconstruction errors with the quantised model
print("Computing int8 training errors (this is the calibration set for threshold)...")
reconstructed_train_int8 = md.predict_tflite(interp_int8_v2, X_train)
err_train_int8 = np.mean((X_train - reconstructed_train_int8) ** 2, axis=(1, 2))

# Derive a fresh threshold from the quantised model's training distribution
threshold_int8 = np.quantile(err_train_int8, 0.99)
print(f"\nKeras threshold (99th pct of float training):  {md.THRESHOLD:.5f}")
print(f"Int8 threshold  (99th pct of int8 training):   {threshold_int8:.5f}")
print(f"Ratio: {threshold_int8 / md.THRESHOLD:.3f}x  (expected ~1.12 from mean error shift)")

# (This cell already ran the part that computed err_train_int8 and threshold_int8.
# We just need the event-level scoring with the moved functions.)

# Ground truth labels for test windows (re-derive in this notebook)
y_test = ev.label_windows_by_end_timestamp(
    test_df['timestamp'].values, anomaly_windows, pp.WINDOW_SIZE
)

test_anomaly_windows = [
    (s, e) for s, e in anomaly_windows
    if s >= pp.DEFAULT_SPLIT.test_start and s < pp.DEFAULT_SPLIT.test_end
]

# Event-level scoring with the int8 model and its own threshold
events_int8 = ev.detect_events(
    err_test_int8_v2, threshold_int8, test_df['timestamp'].values, pp.WINDOW_SIZE
)
scores_int8 = ev.score_events(events_int8, test_anomaly_windows)

print(f"=== Event-level scoring (int8 v2 model, int8-derived threshold {threshold_int8:.5f}) ===")
print(f"  Detected events:  {scores_int8['tp_events'] + scores_int8['fp_events']}")
print(f"  True positives:   {scores_int8['tp_events']}/{len(test_anomaly_windows)}")
print(f"  False alarms:     {scores_int8['fp_events']}")
print(f"  Missed events:    {scores_int8['fn_events']}")
print(f"  Precision:        {scores_int8['precision']:.3f}")
print(f"  Recall:           {scores_int8['recall']:.3f}")
print(f"  Event-level F1:   {scores_int8['f1']:.3f}")
print(f"\nFor reference, Keras float model: P=0.25, R=1.0, F1=0.40")

Computing int8 training errors (this is the calibration set for threshold)...

Keras threshold (99th pct of float training):  0.00879
Int8 threshold  (99th pct of int8 training):   0.00598
Ratio: 0.680x  (expected ~1.12 from mean error shift)
=== Event-level scoring (int8 v2 model, int8-derived threshold 0.00598) ===
  Detected events:  14
  True positives:   2/2
  False alarms:     12
  Missed events:    0
  Precision:        0.143
  Recall:           1.000
  Event-level F1:   0.250

For reference, Keras float model: P=0.25, R=1.0, F1=0.40


In [25]:
def write_c_header(tflite_bytes: bytes, output_path: Path, array_name: str) -> None:
    """Write a TFLite model as a C header file with a byte array.

    The 8-byte alignment is required by TFLite Micro for some target
    architectures, including ARM Cortex-M and ESP32 Xtensa/RISC-V.
    """
    n = len(tflite_bytes)
    guard = array_name.upper() + "_H_"
    lines = [
        f"// Auto-generated from {array_name}.tflite",
        f"// Model size: {n} bytes ({n/1024:.1f} KB)",
        f"",
        f"#ifndef {guard}",
        f"#define {guard}",
        f"",
        f"const unsigned char {array_name}[] __attribute__((aligned(8))) = {{",
    ]
    # Write 12 bytes per line for readability
    for i in range(0, n, 12):
        chunk = tflite_bytes[i:i+12]
        hex_values = ", ".join(f"0x{b:02x}" for b in chunk)
        lines.append(f"  {hex_values},")
    lines.append("};")
    lines.append(f"const unsigned int {array_name}_len = {n};")
    lines.append("")
    lines.append(f"#endif  // {guard}")
    output_path.write_text("\n".join(lines))


# Generate the header
firmware_dir = PROJECT_ROOT / "firmware"
firmware_dir.mkdir(parents=True, exist_ok=True)
header_path = firmware_dir / "autoencoder_model.h"

write_c_header(
    tflite_bytes=tflite_int8_v2_path.read_bytes(),
    output_path=header_path,
    array_name="g_autoencoder_model",
)

print(f"C header written to: {header_path}")
print(f"Header file size:    {header_path.stat().st_size / 1024:.1f} KB")
print(f"Model size in header: {tflite_int8_v2_path.stat().st_size / 1024:.1f} KB")
print()
print("First few lines of the header:")
print("─" * 60)
print("\n".join(header_path.read_text().splitlines()[:8]))

C header written to: D:\dissertation\firmware\autoencoder_model.h
Header file size:    89.5 KB
Model size in header: 14.3 KB

First few lines of the header:
────────────────────────────────────────────────────────────
// Auto-generated from g_autoencoder_model.tflite
// Model size: 14616 bytes (14.3 KB)

#ifndef G_AUTOENCODER_MODEL_H_
#define G_AUTOENCODER_MODEL_H_

const unsigned char g_autoencoder_model[] __attribute__((aligned(8))) = {
  0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00,


In [26]:
# Generate a test input + expected output for ESP32 firmware integration test.
# Pick a window with strong anomalous signal so the reference values are
# meaningful — we want output that's distinct from input.

# Find the window with highest reconstruction error in test set
test_window_idx = int(np.argmax(err_test_int8_v2))
print(f"Using test window index {test_window_idx} (highest int8 reconstruction error)")
print(f"Window end timestamp: {test_df['timestamp'].iloc[test_window_idx + pp.WINDOW_SIZE - 1]}")
print(f"Window reconstruction error: {err_test_int8_v2[test_window_idx]:.5f}")

test_window_float = X_test[test_window_idx]  # shape (60, 1), float32

# Quantise input to int8 using the model's input quantisation params
input_details = interp_int8_v2.get_input_details()[0]
output_details = interp_int8_v2.get_output_details()[0]
in_scale, in_zp = input_details['quantization']
out_scale, out_zp = output_details['quantization']

test_input_int8 = np.clip(
    np.round(test_window_float / in_scale + in_zp), -128, 127
).astype(np.int8)

# Run through TFLite int8 interpreter
interp_int8_v2.set_tensor(input_details['index'], test_input_int8.reshape(1, 60, 1))
interp_int8_v2.invoke()
test_output_int8 = interp_int8_v2.get_tensor(output_details['index']).flatten()

# Compute the expected reconstruction MSE that the firmware should produce.
# This is what the ESP32 will compute from the int8 output, dequantised.
reconstructed_float = (test_output_int8.astype(np.float32) - out_zp) * out_scale
expected_mse = np.mean((test_window_float.flatten() - reconstructed_float) ** 2)
print(f"Expected reconstruction MSE (firmware should produce this): {expected_mse:.5f}")
print(f"Expected confidence score: {(expected_mse - 0.00972) / 0.00972:.3f}")
print(f"(Positive => anomalous; negative => normal)")

# Format as C array initialisers
def format_c_array(values, name, dtype="int8_t"):
    lines = [f"const {dtype} {name}[{len(values)}] = {{"]
    for i in range(0, len(values), 12):
        chunk = values[i:i+12]
        line = "    " + ", ".join(f"{int(v):4d}" for v in chunk) + ","
        lines.append(line)
    lines.append("};")
    return "\n".join(lines)

print("\n" + "=" * 60)
print("Copy these into the firmware:")
print("=" * 60)
print(format_c_array(test_input_int8.flatten(), "kTestInput"))
print()
print(format_c_array(test_output_int8.flatten(), "kExpectedOutput"))

Using test window index 3834 (highest int8 reconstruction error)
Window end timestamp: 2014-02-09 12:25:00
Window reconstruction error: 0.01517
Expected reconstruction MSE (firmware should produce this): 0.01517
Expected confidence score: 0.561
(Positive => anomalous; negative => normal)

Copy these into the firmware:
const int8_t kTestInput[60] = {
    -116, -110, -110, -115, -113, -114, -115, -114, -115, -110, -110, -110,
    -115, -115, -117, -114, -110, -115, -112, -112, -116, -113, -117, -110,
    -115, -112, -111, -114, -116, -116, -112, -113, -116, -117, -112, -111,
    -113, -110, -113, -112, -114, -117, -112, -116, -112, -111, -116, -110,
    -115, -112, -112, -112,  -98,  -72,  -44,  -10,    7,   23,   33,   44,
};

const int8_t kExpectedOutput[60] = {
    -112, -110, -114, -114, -112, -109, -112, -115, -112, -112, -112, -114,
    -114, -113, -116, -117, -111, -111, -111, -114, -113, -111, -113, -116,
    -111, -110, -111, -118, -116, -111, -113, -118, -114, -115, -112, -114,

In [27]:
expected_conf = (expected_mse - 0.00972) / 0.00972

header_lines = [
    "// Auto-generated test reference values for ESP32 integration test",
    "// Source: notebook 05_tflite_conversion.ipynb",
    "// Window: Feb 3 2014 16:45, highest int8 reconstruction error",
    "",
    "#ifndef TEST_REFERENCE_H_",
    "#define TEST_REFERENCE_H_",
    "",
    "#include <stdint.h>",
    "",
    format_c_array(test_input_int8.flatten(), "kTestInput"),
    "",
    format_c_array(test_output_int8.flatten(), "kExpectedOutput"),
    "",
    f"const float kExpectedReconstructionMSE = {expected_mse:.6f}f;",
    f"const float kExpectedConfidence = {expected_conf:.6f}f;",
    "const float kThresholdInt8 = 0.00972f;",
    "",
    "#endif  // TEST_REFERENCE_H_",
]

test_ref_path = PROJECT_ROOT / "firmware" / "tier1_inference" / "main" / "test_reference.h"
test_ref_path.write_text("\n".join(header_lines))
print(f"Saved {test_ref_path}")

Saved D:\dissertation\firmware\tier1_inference\main\test_reference.h


In [28]:
# Inspect the operators in the int8 TFLite model
interp_diag = tf.lite.Interpreter(model_path=str(tflite_int8_v2_path))
interp_diag.allocate_tensors()

ops = interp_diag._get_ops_details()  # private API but stable for this purpose
op_names = sorted(set(op['op_name'] for op in ops))

print(f"Total ops in graph: {len(ops)}")
print(f"Unique op types: {len(op_names)}")
print()
for name in op_names:
    count = sum(1 for op in ops if op['op_name'] == name)
    print(f"  {name:<20} {count}x")

Total ops in graph: 37
Unique op types: 10

  ADD                  1x
  CONCATENATION        2x
  CONV_2D              3x
  DELEGATE             7x
  EXPAND_DIMS          7x
  MAX_POOL_2D          2x
  RESHAPE              9x
  SHAPE                2x
  STRIDED_SLICE        2x
  TRANSPOSE_CONV       2x


D:\dissertation\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [29]:
# Regenerate the C header from the freshly-quantised int8 model
firmware_main = PROJECT_ROOT / "firmware" / "tier1_inference" / "main"
firmware_main.mkdir(parents=True, exist_ok=True)
model_header_path = firmware_main / "autoencoder_model.h"

write_c_header(
    tflite_bytes=tflite_int8_v2_path.read_bytes(),
    output_path=model_header_path,
    array_name="g_autoencoder_model",
)

print(f"Model header written to: {model_header_path}")
print(f"Header size: {model_header_path.stat().st_size / 1024:.1f} KB")
print(f"Embedded model size: {tflite_int8_v2_path.stat().st_size / 1024:.1f} KB")

Model header written to: D:\dissertation\firmware\tier1_inference\main\autoencoder_model.h
Header size: 89.5 KB
Embedded model size: 14.3 KB


In [30]:
# Regenerate the test reference for the new model
test_window_idx = int(np.argmax(err_test_int8_v2))
print(f"Using test window index {test_window_idx}")
print(f"Window end timestamp: {test_df['timestamp'].iloc[test_window_idx + pp.WINDOW_SIZE - 1]}")
print(f"Window int8 reconstruction error: {err_test_int8_v2[test_window_idx]:.5f}")

test_window_float = X_test[test_window_idx]
input_details = interp_int8_v2.get_input_details()[0]
output_details = interp_int8_v2.get_output_details()[0]
in_scale, in_zp = input_details['quantization']
out_scale, out_zp = output_details['quantization']

test_input_int8 = np.clip(
    np.round(test_window_float / in_scale + in_zp), -128, 127
).astype(np.int8)

interp_int8_v2.set_tensor(input_details['index'], test_input_int8.reshape(1, 60, 1))
interp_int8_v2.invoke()
test_output_int8 = interp_int8_v2.get_tensor(output_details['index']).flatten()

reconstructed_float = (test_output_int8.astype(np.float32) - out_zp) * out_scale
expected_mse = np.mean((test_window_float.flatten() - reconstructed_float) ** 2)
expected_conf = (expected_mse - float(threshold_int8)) / float(threshold_int8)

print(f"Expected reconstruction MSE: {expected_mse:.5f}")
print(f"Expected confidence score: {expected_conf:.3f}")

def format_c_array(values, name, dtype="int8_t"):
    lines = [f"const {dtype} {name}[{len(values)}] = {{"]
    for i in range(0, len(values), 12):
        chunk = values[i:i+12]
        line = "    " + ", ".join(f"{int(v):4d}" for v in chunk) + ","
        lines.append(line)
    lines.append("};")
    return "\n".join(lines)

header_lines = [
    "// Auto-generated test reference values for ESP32 integration test",
    "// Source: notebook 05_tflite_conversion.ipynb",
    f"// Test window index: {test_window_idx}",
    f"// Window end: {test_df['timestamp'].iloc[test_window_idx + pp.WINDOW_SIZE - 1]}",
    "",
    "#ifndef TEST_REFERENCE_H_",
    "#define TEST_REFERENCE_H_",
    "",
    "#include <stdint.h>",
    "",
    format_c_array(test_input_int8.flatten(), "kTestInput"),
    "",
    format_c_array(test_output_int8.flatten(), "kExpectedOutput"),
    "",
    f"const float kExpectedReconstructionMSE = {expected_mse:.6f}f;",
    f"const float kExpectedConfidence = {expected_conf:.6f}f;",
    f"const float kThresholdInt8 = {float(threshold_int8):.6f}f;",
    "",
    "#endif  // TEST_REFERENCE_H_",
]

test_ref_path = firmware_main / "test_reference.h"
test_ref_path.write_text("\n".join(header_lines))
print(f"\nSaved {test_ref_path}")

Using test window index 3834
Window end timestamp: 2014-02-09 12:25:00
Window int8 reconstruction error: 0.01517
Expected reconstruction MSE: 0.01517
Expected confidence score: 1.536

Saved D:\dissertation\firmware\tier1_inference\main\test_reference.h
